In [26]:
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Create a SparkSession and run locally (local[*]) for notebooks
spark = SparkSession.builder\
    .appName("students_and_examinations")\
    .getOrCreate()

data = [[1, 'Alice'], [2, 'Bob'], [13, 'John'], [6, 'Alex']]
students = pd.DataFrame(data, columns=['student_id', 'student_name']).astype({'student_id':'Int64', 'student_name':'object'})
data = [['Math'], ['Physics'], ['Programming']]
subjects = pd.DataFrame(data, columns=['subject_name']).astype({'subject_name':'object'})
data = [[1, 'Math'], [1, 'Physics'], [1, 'Programming'], [2, 'Programming'], [1, 'Physics'], [1, 'Math'], [13, 'Math'], [13, 'Programming'], [13, 'Physics'], [2, 'Math'], [1, 'Math']]
examinations = pd.DataFrame(data, columns=['student_id', 'subject_name']).astype({'student_id':'Int64', 'subject_name':'object'})

# Convert pandas DataFrames to Spark DataFrames
students_spark = spark.createDataFrame(students)
subjects_spark = spark.createDataFrame(subjects)
examinations_spark = spark.createDataFrame(examinations)

students_spark.show()
subjects_spark.show()
examinations_spark.show()

students_subjects_df = students_spark.join(subjects_spark, how='cross')

students_subjects_df.show()
'''
joined_df = students_subjects_df.join(examinations_spark, on=['student_id', 'subject_name'], how='left')\
    .select(students_subjects_df['student_id'], students_subjects_df['student_name'],\
             students_subjects_df['subject_name'],examinations_spark['subject_name'].alias('exam_student_id'))
print("Joined DataFrame :")
joined_df.show()


final_df = joined_df\
    .groupby('student_id', 'student_name', 'subject_name')\
    .agg(F.count('exam_student_id').alias('examination_count'))\
    .select('student_id', 'student_name', 'subject_name', 'examination_count')\
    .orderBy('student_id', 'subject_name')
'''

final_df = (
    students_subjects_df
    .join(examinations_spark, on=['student_id', 'subject_name'], how='left')
    .groupby('student_id', 'student_name', 'subject_name')
    .agg(F.count(examinations_spark['subject_name']).alias('examination_count'))
    .orderBy('student_id', 'subject_name')
)

final_df2 = students_subjects_df.alias('ss').join(
    examinations_spark.alias('es'),
    on=['student_id', 'subject_name'],
    how='left'
).groupby('ss.student_id', 'ss.student_name', 'ss.subject_name'
).agg(F.sum(F.when(F.col('es.student_id').isNotNull(), 1).otherwise(0)).alias('examination_count'))

print("Final DataFrame :")
final_df.show()

print("Final DataFrame 2 :")
final_df2.show()



# Stop the Spark session
spark.stop()

+----------+------------+
|student_id|student_name|
+----------+------------+
|         1|       Alice|
|         2|         Bob|
|        13|        John|
|         6|        Alex|
+----------+------------+

+------------+
|subject_name|
+------------+
|        Math|
|     Physics|
| Programming|
+------------+

+----------+------------+
|student_id|subject_name|
+----------+------------+
|         1|        Math|
|         1|     Physics|
|         1| Programming|
|         2| Programming|
|         1|     Physics|
|         1|        Math|
|        13|        Math|
|        13| Programming|
|        13|     Physics|
|         2|        Math|
|         1|        Math|
+----------+------------+



+----------+------------+------------+
|student_id|student_name|subject_name|
+----------+------------+------------+
|         1|       Alice|        Math|
|         1|       Alice|     Physics|
|         1|       Alice| Programming|
|         2|         Bob|        Math|
|         2|         Bob|     Physics|
|         2|         Bob| Programming|
|        13|        John|        Math|
|        13|        John|     Physics|
|        13|        John| Programming|
|         6|        Alex|        Math|
|         6|        Alex|     Physics|
|         6|        Alex| Programming|
+----------+------------+------------+

Final DataFrame :


+----------+------------+------------+-----------------+
|student_id|student_name|subject_name|examination_count|
+----------+------------+------------+-----------------+
|         1|       Alice|        Math|                3|
|         1|       Alice|     Physics|                2|
|         1|       Alice| Programming|                1|
|         2|         Bob|        Math|                1|
|         2|         Bob|     Physics|                0|
|         2|         Bob| Programming|                1|
|         6|        Alex|        Math|                0|
|         6|        Alex|     Physics|                0|
|         6|        Alex| Programming|                0|
|        13|        John|        Math|                1|
|        13|        John|     Physics|                1|
|        13|        John| Programming|                1|
+----------+------------+------------+-----------------+

Final DataFrame 2 :


+----------+------------+------------+-----------------+
|student_id|student_name|subject_name|examination_count|
+----------+------------+------------+-----------------+
|         1|       Alice|        Math|                3|
|         1|       Alice|     Physics|                2|
|         1|       Alice| Programming|                1|
|         2|         Bob|        Math|                1|
|         2|         Bob|     Physics|                0|
|         2|         Bob| Programming|                1|
|        13|        John|        Math|                1|
|        13|        John|     Physics|                1|
|        13|        John| Programming|                1|
|         6|        Alex|        Math|                0|
|         6|        Alex|     Physics|                0|
|         6|        Alex| Programming|                0|
+----------+------------+------------+-----------------+

